# Chapter 7 — pandas II: Cleaning Messy Indian Data

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch07_cleaning_messy_data.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 7.1 A raw table from the API

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

from indiadata import mospi, mospi_nas, NAS_REVISIONS

raw = mospi("nas/getNASData", indicator_code=5, base_year="2011-12",
            series="Current", frequency_code=1)
print(raw.shape)
raw.head()

In [ ]:
raw.dtypes

In [ ]:
print(raw["current_price"].head(3).sum())

In [ ]:
gdp = raw.drop(columns=["base_year", "series", "indicator", "frequency", "unit"])
for col in ["current_price", "constant_price"]:
    gdp[col] = pd.to_numeric(gdp[col], errors="coerce")
print(gdp.dtypes)
print("\nMissing values per column:")
print(gdp.isna().sum())

In [ ]:
graw = mospi("nas/getNASData", indicator_code=23, base_year="2011-12",
             series="Current", frequency_code=1)
graw["constant_price"] = pd.to_numeric(graw["constant_price"], errors="coerce")
missing = graw[graw["constant_price"].isna()]
print(f"{len(missing)} missing values, all in year(s):",
      missing["year"].unique().tolist())
gdp = gdp.drop(columns="quarter")

## 7.2 Fiscal years: from "2023-24" to dates

In [ ]:
parts = gdp["year"].str.split("-", expand=True)
parts.head(3)

In [ ]:
def fiscal_year(label: pd.Series) -> pd.DataFrame:
    """Turn labels like '2023-24' into start year, end year and dates."""
    start = label.str[:4].astype(int)
    end2 = label.str[-2:].astype(int)
    assert ((start + 1) % 100 == end2).all(), "label is not of the form YYYY-YY"
    return pd.DataFrame({
        "fy": start,
        "fy_end": start + 1,
        "fy_start_date": pd.to_datetime(start.astype(str) + "-04-01"),
        "fy_end_date": pd.to_datetime((start + 1).astype(str) + "-03-31"),
    }, index=label.index)


gdp = gdp.join(fiscal_year(gdp["year"]))
gdp[["year", "fy", "fy_end", "fy_start_date", "fy_end_date"]].head(3)

In [ ]:
q = mospi("nas/getNASData", indicator_code=5, base_year="2011-12",
          series="Current", frequency_code=2)
q["constant_price"] = pd.to_numeric(q["constant_price"])
q_num = q["quarter"].str[1].astype(int)                  # "Q3" -> 3
start_month = 4 + 3 * (q_num - 1)                        # Q1 -> 4 (April)
cal_year = q["year"].str[:4].astype(int) + (start_month > 12)
cal_month = (start_month - 1) % 12 + 1
q["date"] = pd.PeriodIndex.from_fields(year=cal_year, month=cal_month,
                                       freq="M").asfreq("Q").to_timestamp()
q = q.sort_values("date")
q[["year", "quarter", "date", "constant_price"]].tail(5)

## 7.3 Revisions: the same year, many numbers

In [ ]:
gdp.groupby("year").size().tail(8)

In [ ]:
gdp[gdp["year"] == "2021-22"][["year", "revision", "constant_price"]]

In [ ]:
print(NAS_REVISIONS)
gdp["revision"] = pd.Categorical(gdp["revision"], categories=NAS_REVISIONS,
                                 ordered=True)
assert gdp["revision"].notna().all(), "a revision name is not in the list"

In [ ]:
latest = (gdp.sort_values(["fy", "revision"])
             .drop_duplicates(subset="year", keep="last"))
latest[["year", "revision", "constant_price"]].tail(6)

In [ ]:
ours = latest.set_index("year")["constant_price"]
theirs = mospi_nas("GDP").set_index("year")["constant_price"]
assert ours.sort_index().equals(theirs.sort_index().astype(ours.dtype))
print("Same as mospi_nas('GDP'): yes,", len(ours), "years")

In [ ]:
first = (gdp.sort_values(["fy", "revision"])
            .drop_duplicates(subset="year", keep="first")
            .set_index("year"))
rev = pd.DataFrame({
    "first estimate": first["revision"].astype(str),
    "latest estimate": latest.set_index("year")["revision"].astype(str),
    "revision of level, %": 100 * (ours / first["constant_price"] - 1),
}).loc[lambda d: d["first estimate"] != d["latest estimate"]]
rev.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#c55a11" if v < 0 else "#1f4e79" for v in rev["revision of level, %"]]
ax.bar(rev.index, rev["revision of level, %"], color=colors)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Latest minus first estimate, %")
ax.set_title("How much has real GDP been revised since its first estimate?",
             loc="left", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout(rect=(0, 0.09, 1, 1))
fig.text(0.01, 0.005, "Constant 2011–12 prices. First estimate is the earliest "
         "vintage in the API (Advance Estimates from 2014–15).\nSource: MoSPI, "
         "National Accounts Statistics, via eSankhyiki API; author's calculation.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
r = rev["revision of level, %"]
print(f"Years revised: {len(r)};  upward: {(r > 0).sum()},  downward: {(r < 0).sum()}")
print(f"Largest upward revision: {r.idxmax()} ({r.max():+.2f}%)")
print(f"Largest downward revision: {r.idxmin()} ({r.min():+.2f}%)")
print(f"Average absolute revision: {r.abs().mean():.2f}%")

## 7.4 Duplicate rows

In [ ]:
gsdp = mospi_nas("GSDP")[["state", "year", "fy", "current_price",
                          "constant_price"]]
oops = pd.concat([gsdp, gsdp.tail(30)], ignore_index=True)
print("rows:", len(oops))
print("exact duplicate rows:", oops.duplicated().sum())
print("duplicate state-years:", oops.duplicated(subset=["state", "fy"]).sum())

In [ ]:
clean = oops.drop_duplicates()
assert not clean.duplicated(subset=["state", "fy"]).any()
print("rows after drop_duplicates:", len(clean))
del oops, clean

## 7.5 Wide and long

In [ ]:
wide = gsdp.pivot(index="state", columns="year", values="constant_price")
wide = wide.reset_index()
wide.iloc[:4, [0, 1, 2, -2, -1]]

In [ ]:
long = wide.melt(id_vars="state", var_name="year", value_name="constant_price")
print(wide.shape, "->", long.shape)
long.head(3)

In [ ]:
stacked = gsdp.melt(id_vars=["state", "fy"],
                    value_vars=["current_price", "constant_price"],
                    var_name="prices", value_name="gsdp")
stacked["prices"] = stacked["prices"].str.replace("_price", "")
stacked.sort_values(["state", "fy"]).head(4)

In [ ]:
back = stacked.pivot(index=["state", "fy"], columns="prices", values="gsdp")
back.head(3)

## 7.6 Codes and the leading-zero trap

In [ ]:
from indiadata import read_plfs, labour_status, labour_rates

p = read_plfs("perv1", columns=["sex", "age", "pas", "sas", "ind_pas"])
print(f"{len(p):,} persons; memory {p.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print("ind_pas dtype:", p["ind_pas"].dtype)
p["ind_pas"].dropna().astype(int).astype(str).str.len().value_counts().sort_index()

In [ ]:
code = p["ind_pas"].dropna().astype(int)
wrong = code.astype(str).str[:2]                  # "1121" -> "11" (beverages!)
right = code.astype(str).str.zfill(5).str[:2]     # "01121" -> "01" (crops)
print("rows given a wrong 2-digit division:", (wrong != right).sum())
pd.DataFrame({"stored": code, "wrong": wrong, "right": right}).head(3)

In [ ]:
codes_text = pd.DataFrame({"st": ["01", "09", "27"], "x": [1, 2, 3]})
codes_num = pd.DataFrame({"st": [1, 9, 27], "y": [4, 5, 6]})
try:
    codes_text.merge(codes_num, on="st")
except ValueError as err:
    print("ValueError:", err)

## 7.7 PLFS state estimates

In [ ]:
p["status"] = labour_status(p["pas"], p["sas"])
a = p[p["age"] >= 15]
plfs = a.groupby("st").apply(labour_rates)[["LFPR", "UR"]]
plfs["female_LFPR"] = a[a["sex"] == 2].groupby("st").apply(labour_rates)["LFPR"]
plfs["sample_15plus"] = a.groupby("st").size()           # persons interviewed
plfs["pop_15plus"] = a.groupby("st")["weight"].sum()     # persons represented
print("All-India check:", labour_rates(a).round(1).to_dict())
del p, a                                   # free memory: we only need the totals
plfs.head()

In [ ]:
layout = next(Path("data/plfs").glob("Data_Layout*.xlsx"))
plfs_states = pd.read_excel(layout, sheet_name="State code", header=1)
plfs_states.columns = ["st", "plfs_name"]
plfs_states = plfs_states.dropna().astype({"st": int})
plfs_states["plfs_name"] = plfs_states["plfs_name"].str.strip()
plfs = plfs.reset_index().merge(plfs_states, on="st", how="left",
                                validate="one_to_one")
assert plfs["plfs_name"].notna().all()
print(len(plfs), "states and UTs in PLFS")

## 7.8 Same state, different names

In [ ]:
g23 = gsdp[gsdp["fy"] == 2023].rename(columns={"state": "nas_name"})
naive = g23.merge(plfs, left_on="nas_name", right_on="plfs_name",
                  how="outer", indicator=True)
naive["_merge"].value_counts()

In [ ]:
print("GSDP only:", sorted(naive.loc[naive["_merge"] == "left_only", "nas_name"]))
print("PLFS only:", sorted(naive.loc[naive["_merge"] == "right_only", "plfs_name"]))

In [ ]:
def normalise(name: pd.Series) -> pd.Series:
    return (name.str.lower()
                .str.replace("&", " and ", regex=False)
                .str.replace(".", " ", regex=False)
                .str.replace(r"\s+", " ", regex=True)
                .str.strip())


print(normalise(pd.Series(["Tamil Nadu", "Tamilnadu", "Jammu & Kashmir",
                           "Andaman & N. Island"])).tolist())

In [ ]:
CROSSWALK = {                          # PLFS layout name -> NAS name
    "Tamilnadu": "Tamil Nadu",
    "Puduchery": "Puducherry",
    "Andaman & N. Island": "Andaman & Nicobar Islands",
    "D & N. Haveli & Daman & Diu": "Dadra & Nagar Haveli and Daman & Diu",
}
plfs["state"] = plfs["plfs_name"].replace(CROSSWALK)
g23 = g23.rename(columns={"nas_name": "state"})

merged = g23.merge(plfs, on="state", how="outer", indicator=True,
                   validate="one_to_one")
print(merged["_merge"].value_counts().to_string())
print("\nStill unmatched:",
      merged.loc[merged["_merge"] != "both", ["state", "_merge"]]
            .to_string(index=False))

## 7.9 New states and union territories

In [ ]:
jk = gsdp[gsdp["state"] == "Jammu & Kashmir"].set_index("year")
jk_growth = 100 * jk[["current_price", "constant_price"]].pct_change()
jk_growth.loc["2017-18":"2021-22"].round(1)

In [ ]:
jl = plfs[plfs["plfs_name"].isin(["Jammu & Kashmir", "Ladakh"])].set_index("plfs_name")
combined = np.average(jl["LFPR"], weights=jl["pop_15plus"])
print(jl[["LFPR", "pop_15plus", "sample_15plus"]].round(1).to_string())
print(f"J&K and Ladakh combined LFPR: {combined:.2f}%  "
      f"(J&K alone {jl.loc['Jammu & Kashmir', 'LFPR']:.2f}%)")
print(f"Ladakh's share of the two UTs' population aged 15+: "
      f"{100 * jl.loc['Ladakh', 'pop_15plus'] / jl['pop_15plus'].sum():.1f}%")

## 7.10 Checks with `assert`

In [ ]:
state_23 = merged.drop(columns="_merge").copy()
state_23["has_gsdp"] = state_23["constant_price"].notna()

assert state_23["state"].is_unique, "a state appears twice"
assert state_23["st"].notna().all(), "a state has no PLFS code"
assert state_23["LFPR"].between(0, 100).all(), "LFPR outside 0-100"
assert state_23["UR"].between(0, 100).all(), "UR outside 0-100"
assert state_23["has_gsdp"].sum() == 30, "expected 30 states with GSDP"
assert (state_23["sample_15plus"] > 0).all()
print("All checks passed.", len(state_23), "rows,",
      state_23["has_gsdp"].sum(), "with GSDP.")

## 7.11 What the merged table shows

In [ ]:
w = gsdp.pivot(index="state", columns="fy", values="constant_price")
cagr = (100 * ((w[2024] / w[2011]) ** (1 / 13) - 1)).rename("real_CAGR")
xs = state_23.merge(cagr, left_on="state", right_index=True, how="inner")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(xs["real_CAGR"], xs["female_LFPR"], s=40, color="#1f4e79", alpha=0.8)
for _, r in xs.iterrows():
    if r["state"] in ("Mizoram", "Gujarat", "Goa", "Bihar", "Himachal Pradesh",
                      "Delhi", "Meghalaya", "Karnataka", "Uttar Pradesh"):
        ax.annotate(r["state"], (r["real_CAGR"], r["female_LFPR"]),
                    xytext=(5, 3), textcoords="offset points", fontsize=8.5)
ax.set_xlabel("Real GSDP growth, CAGR 2011–12 to 2024–25, %")
ax.set_ylabel("Female LFPR 2023–24, % of women 15+")
ax.set_title("Growth and women's participation across states",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.04, "Usual status (ps+ss). 30 states/UTs with GSDP data. "
         "Source: MoSPI NAS (GSDP); author's calculation from PLFS 2023–24 "
         "unit-level data.", fontsize=8, color="grey")
plt.show()

print(f"Correlation: {xs['real_CAGR'].corr(xs['female_LFPR']):.2f} "
      f"({len(xs)} states/UTs)")

## 7.12 Save a tidy table

In [ ]:
nsdp = mospi_nas("NSDP")[["state", "fy", "current_price", "constant_price"]]
panel = (gsdp.merge(nsdp, on=["state", "fy"], how="outer", validate="one_to_one",
                    suffixes=("", "_nsdp"))
             .rename(columns={"current_price": "gsdp_current",
                              "constant_price": "gsdp_constant",
                              "current_price_nsdp": "nsdp_current",
                              "constant_price_nsdp": "nsdp_constant"}))
panel = panel.join(fiscal_year(panel["year"]).drop(columns="fy"))
panel = panel.sort_values(["state", "fy"])
panel["gsdp_real_growth"] = 100 * panel.groupby("state")["gsdp_constant"].pct_change()

assert not panel.duplicated(subset=["state", "fy"]).any()
assert panel["state"].nunique() == 30 and panel["fy"].between(2011, 2025).all()

CLEAN = Path("data/clean")
CLEAN.mkdir(parents=True, exist_ok=True)
panel.to_csv(CLEAN / "state_gsdp_nsdp_panel.csv", index=False)

cross = state_23[["state", "st", "plfs_name", "current_price", "constant_price",
                  "LFPR", "UR", "female_LFPR", "sample_15plus",
                  "pop_15plus"]].rename(
    columns={"current_price": "gsdp_current_2023_24",
             "constant_price": "gsdp_constant_2023_24",
             "st": "plfs_state_code"})
cross.to_csv(CLEAN / "state_2023_24_gsdp_plfs.csv", index=False)

print(panel.shape, cross.shape)
panel.head(3)

## 7.13 A manual download: the RBI Handbook

In [ ]:
HANDBOOK = Path("data/manual/rbi_handbook_states_gsdp_constant.xlsx")
if HANDBOOK.exists():
    hb = pd.read_excel(HANDBOOK, header=None)
    print(hb.shape)
    display(hb.head(12))         # look first: titles, notes, merged headers
else:
    print(f"{HANDBOOK} not found: skipping. See Exercise 9 to download it.")

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading